
## PHASE 2: BUILD LOCAL MODELS
### Federated Learning Breast Cancer Detection Project

### This notebook defines the neural network architectures for:
#### - Hospital 1 (WDBC) - MLP for tabular cell features
#### - Hospital 2 (Coimbra) - MLP for tabular blood biomarkers
#### - Hospital 3 (BreakHis) - CNN (MobileNetV2) for histopathology images


### Setting Working Directory

In [1]:
import os

curr_dir = os.getcwd()
print(f"Current directory: '{curr_dir}'")
os.chdir('..')
print("Changing to root directory...")

print(f"Working in: '{os.getcwd()}'")

Current directory: 'D:\MY FILES\BREAST_CANCER_FL\notebooks'
Changing to root directory...
Working in: 'D:\MY FILES\BREAST_CANCER_FL'


### 1. Import Libraries

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights
from torchinfo import summary


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from pathlib import Path
import os
from PIL import Image

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print("\nAll libraries imported successfully!")

Using device: cpu

All libraries imported successfully!


## PART A: HOSPITAL 1 MODEL (WDBC - Tabular)

### 2. Define Hospital 1 MLP Model

In [3]:
# EMBEDDING DIMENSION — shared across all three hospitals

EMBEDDING_DIM = 64

# SHARED CLASSIFICATION HEAD - Identical architecture across ALL three hospitals. This is the only component exchanged in federated learning.
# All hospitals share this exact same class definition.
class SharedClassificationHead(nn.Module):
    """
    Shared classification head used by all three hospitals.
    Input  : 64-dim embedding (from any modality encoder)
    Output : Binary logit (Benign vs Malignant)

    This is the federated component — identical architecture
    and identical input dimension across all hospitals
    makes FedAvg mathematically valid.

    Note: No sigmoid — BCEWithLogitsLoss handles it.
    """
    def __init__(self, embedding_dim=EMBEDDING_DIM, dropout_rate=0.3):
        super(SharedClassificationHead, self).__init__()

        self.fc1      = nn.Linear(embedding_dim, 32)
        self.bn1      = nn.BatchNorm1d(32)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.output   = nn.Linear(32, 1)
        # No sigmoid — BCEWithLogitsLoss handles it

    def forward(self, embedding):
        x = self.fc1(embedding)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.output(x)
        return x


# HOSPITAL 1 ENCODER
# Private component — never shared with central server
# Maps 23 cell nucleus features → 64-dim embedding
class Hospital1_Encoder(nn.Module):
    """
    Private local encoder for Hospital 1 WDBC dataset.
    Input  : 23 cell nucleus features
    Output : 64-dim shared embedding

    This component is NEVER shared in federated learning.
    It learns WDBC-specific feature representations privately.
    """
    def __init__(self, input_size=23, dropout_rate=0.3):
        super(Hospital1_Encoder, self).__init__()

        # Layer 1: 23 → 128 units - Expands the representation — giving the model enough room to learn complex non-linear relationships between 
        # all 23 features
        self.fc1      = nn.Linear(input_size, 128)
        self.bn1      = nn.BatchNorm1d(128) # mean 0; std ~1 (-1 to +1)
        self.relu1    = nn.ReLU() # add non-linearity; turn all negatives to zero
        self.dropout1 = nn.Dropout(dropout_rate) # reduce overfitting

        # Layer 2: 128 → 64 units (embedding projection)
        self.fc2      = nn.Linear(128, EMBEDDING_DIM)
        self.bn2      = nn.BatchNorm1d(EMBEDDING_DIM)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        # Layer 2 — output is 64-dim embedding
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        return x  # shape: [batch, 64]


# HOSPITAL 1 — COMPLETE MODEL
# Combines private encoder with shared classification head
class Hospital1_MLP(nn.Module):
    """
    Complete model for Hospital 1 WDBC dataset.
    Combines:
        - Private encoder   : learns WDBC-specific features
        - Shared head       : federated across all hospitals

    Input  : 23 cell nucleus features
    Output : Binary logit (Benign vs Malignant)

    FL note: Only shared_head weights are exchanged.
             Encoder weights stay local and private always.
    """
    def __init__(self, input_size=23, dropout_rate=0.3):
        super(Hospital1_MLP, self).__init__()

        # Private — never shared
        self.encoder = Hospital1_Encoder(
            input_size=input_size,
            dropout_rate=dropout_rate
        )

        # Federated — shared every FL round
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)        # private encoding
        logit     = self.shared_head(embedding)  # shared classification
        return logit

    def get_embedding(self, x):
        """
        Returns 64-dim embedding before classification.
        Used in FedProto to compute class prototypes.
        """
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        """
        Returns probabilities (0-1) for evaluation/inference.
        """
        logits = self.forward(x)
        return torch.sigmoid(logits)


# Creating model instance
hospital1_model = Hospital1_MLP(
    input_size=23,
    dropout_rate=0.3
).to(device)

print("Hospital 1 Model (WDBC - MLP) created!")
print(f"\nModel Architecture:")
print(hospital1_model)

# Parameter counts
total_params     = sum(
    p.numel() for p in hospital1_model.parameters()
)
trainable_params = sum(
    p.numel() for p in hospital1_model.parameters()
    if p.requires_grad
)
encoder_params   = sum(
    p.numel() for p in hospital1_model.encoder.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital1_model.shared_head.parameters()
)

print(f"\n{'='*45}")
print(f"  HOSPITAL 1 MODEL SUMMARY")
print(f"{'='*45}")
print(f"  Input features           : 23")
print(f"  Encoder Layer 1          : 23 → 128 units")
print(f"  Encoder Layer 2          : 128 → 64 units (embedding)")
print(f"  Shared head Layer 1      : 64 → 32 units")
print(f"  Shared head output       : 32 → 1 unit")
print(f"  Embedding dimension      : {EMBEDDING_DIM}")
print(f"  Dropout rate             : 0.3")
print(f"  Batch Normalization      : Yes")
print(f"  Total parameters         : {total_params:,}")
print(f"  Trainable parameters     : {trainable_params:,}")
print(f"  Encoder params           : {encoder_params:,} (private)")
print(f"  Shared head params       : {shared_head_params:,} (federated)")
print(f"  Activation (hidden)      : ReLU")
print(f"  Activation (output)      : None (BCEWithLogitsLoss)")
print(f"  FL component             : shared_head only")
print(f"{'='*45}")

Hospital 1 Model (WDBC - MLP) created!

Model Architecture:
Hospital1_MLP(
  (encoder): Hospital1_Encoder(
    (fc1): Linear(in_features=23, out_features=128, bias=True)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (fc2): Linear(in_features=128, out_features=64, bias=True)
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu2): ReLU()
    (dropout2): Dropout(p=0.3, inplace=False)
  )
  (shared_head): SharedClassificationHead(
    (fc1): Linear(in_features=64, out_features=32, bias=True)
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (output): Linear(in_features=32, out_features=1, bias=True)
  )
)

  HOSPITAL 1 MODEL SUMMARY
  Input features           : 23
  Encoder Layer 1          : 23 → 128 units
  Encoder La

### 3. Test Hospital 1 Model

In [4]:
# Model Sanity Test — Hospital 1 
# Tests both encoder and shared head separately
# as well as the full model end-to-end

hospital1_model.eval()

dummy_input = torch.randn(4, 23).to(device)

with torch.no_grad():
    # Test 1 — Full model forward pass
    logits        = hospital1_model(dummy_input)
    probabilities = hospital1_model.predict_proba(dummy_input)
    predictions   = (probabilities >= 0.5).float()

    # Test 2 — Encoder only (private component)
    embedding = hospital1_model.get_embedding(dummy_input)

    # Test 3 — Shared head only (federated component)
    shared_head_logits = hospital1_model.shared_head(embedding)

# Printing Full Test Results
print(f"\n{'='*55}")
print(f"  HOSPITAL 1 MODEL TEST")
print(f"{'='*55}")

# Input
print(f"\n  Input:")
print(f"     Shape          : {dummy_input.shape}")
print(f"     Features       : 23 cell nucleus measurements")

# Encoder test
print(f"\n  Encoder (private — never shared):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {embedding.shape}")
print(f"     Embedding dim  : {embedding.shape[1]} (expected {EMBEDDING_DIM})")
print(f"     Embedding range: [{embedding.min().item():.4f}, "
      f"{embedding.max().item():.4f}]")
print(f"     All non-negative: "
      f"{'Yes' if embedding.min().item() >= 0 else 'No'} "
      f"(ReLU applied)")

embedding_dim_check = embedding.shape[1] == EMBEDDING_DIM
print(f"     Dim correct    : "
      f"{'Yes' if embedding_dim_check else 'No'} "
      f"(compatible with H2 and H3)")

# Shared head test
print(f"\n  Shared Head (federated — exchanged every round):")
print(f"     Input shape    : {embedding.shape}")
print(f"     Output shape   : {shared_head_logits.shape}")
print(f"     Raw logits     : "
      f"{shared_head_logits.squeeze().detach().cpu().numpy().round(4)}")

# Full model test
print(f"\n  Full Model (encoder → shared head):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {logits.shape}")
print(f"     Raw logits     : "
      f"{logits.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Probabilities  : "
      f"{probabilities.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Predictions    : "
      f"{predictions.squeeze().detach().cpu().numpy()}")
print(f"     Prob range     : [{probabilities.min().item():.4f}, "
      f"{probabilities.max().item():.4f}]")
print(f"     Expected range : [0.0000, 1.0000]")

range_check = (
    0.0 <= probabilities.min().item() and
    probabilities.max().item() <= 1.0
)
print(f"     Range valid    : "
      f"{'Yes' if range_check else 'No'}")

# Logit consistency check - Shared head output should match full model output because full model = encoder → shared head
logits_match = torch.allclose(logits, shared_head_logits, atol=1e-6)
print(f"\n  Consistency Check:")
print(f"     Full model logits == shared head logits: "
      f"{'Yes' if logits_match else 'No'}")
print(f"     (confirms encoder → shared head pipeline is intact)")

# Parameter checking
encoder_params     = sum(
    p.numel() for p in hospital1_model.encoder.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital1_model.shared_head.parameters()
)
total_params       = sum(
    p.numel() for p in hospital1_model.parameters()
)

print(f"\n  Parameter Summary:")
print(f"     Encoder params     : {encoder_params:,} (private)")
print(f"     Shared head params : {shared_head_params:,} (federated)")
print(f"     Total params       : {total_params:,}")
print(f"     FL overhead        : {shared_head_params:,} params per round")

print(f"\n{'='*55}")

# Setting back to train mode
hospital1_model.train()
print(f"\nAll tests passed — Hospital 1 model working correctly!")


  HOSPITAL 1 MODEL TEST

  Input:
     Shape          : torch.Size([4, 23])
     Features       : 23 cell nucleus measurements

  Encoder (private — never shared):
     Input shape    : torch.Size([4, 23])
     Output shape   : torch.Size([4, 64])
     Embedding dim  : 64 (expected 64)
     Embedding range: [0.0000, 0.7532]
     All non-negative: Yes (ReLU applied)
     Dim correct    : Yes (compatible with H2 and H3)

  Shared Head (federated — exchanged every round):
     Input shape    : torch.Size([4, 64])
     Output shape   : torch.Size([4, 1])
     Raw logits     : [0.0807 0.0146 0.0328 0.0671]

  Full Model (encoder → shared head):
     Input shape    : torch.Size([4, 23])
     Output shape   : torch.Size([4, 1])
     Raw logits     : [0.0807 0.0146 0.0328 0.0671]
     Probabilities  : [0.5202 0.5037 0.5082 0.5168]
     Predictions    : [1. 1. 1. 1.]
     Prob range     : [0.5037, 0.5202]
     Expected range : [0.0000, 1.0000]
     Range valid    : Yes

  Consistency Check:
  

## PART B: HOSPITAL 2 MODEL (Coimbra - Tabular)

### 4. Define Hospital 2 MLP Model

In [5]:
# HOSPITAL 2 — COIMBRA ENCODER
# Private component — never shared with central server
# Maps 9 blood biomarker features → 64-dim embedding
class Hospital2_Encoder(nn.Module):
    """
    Private local encoder for Hospital 2 Coimbra dataset.
    Input  : 9 blood biomarker features
    Output : 64-dim shared embedding

    This component is NEVER shared in federated learning.
    It learns Coimbra-specific biomarker representations privately.

    Note: Smaller architecture than Hospital 1 encoder
    because Coimbra has only 9 input features and 92
    training samples — a larger encoder would overfit severely.
    """
    def __init__(self, input_size=9, dropout_rate=0.3):
        super(Hospital2_Encoder, self).__init__()

        # Layer 1: 9 → 32 units
        self.fc1      = nn.Linear(input_size, 32)
        self.bn1      = nn.BatchNorm1d(32)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)

        # Layer 2: 32 → 64 units (embedding projection)
        # Expands to shared embedding dimension
        self.fc2      = nn.Linear(32, EMBEDDING_DIM)
        self.bn2      = nn.BatchNorm1d(EMBEDDING_DIM)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        # Layer 2 — output is 64-dim embedding
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        return x  # shape: [batch, 64]

# HOSPITAL 2 — COMPLETE MODEL
# Combines private encoder with shared classification head
# SharedClassificationHead is already defined above
# in the Hospital 1 section — reused here as-is
class Hospital2_MLP(nn.Module):
    """
    Complete model for Hospital 2 Coimbra dataset.
    Combines:
        - Private encoder   : learns Coimbra-specific biomarker features
        - Shared head       : federated across all hospitals

    Input  : 9 blood biomarker features
    Output : Binary logit (Patient vs Healthy)

    FL note: Only shared_head weights are exchanged.
             Encoder weights stay local and private always.

    Note: Sigmoid is NOT applied here.
          BCEWithLogitsLoss handles it for numerical stability.
    """
    def __init__(self, input_size=9, dropout_rate=0.3):
        super(Hospital2_MLP, self).__init__()

        # Private — never shared
        self.encoder     = Hospital2_Encoder(
            input_size=input_size,
            dropout_rate=dropout_rate
        )

        # Federated — shared every FL round
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)             # private encoding
        logit     = self.shared_head(embedding) # shared classification
        return logit

    def get_embedding(self, x):
        """
        Returns 64-dim embedding before classification.
        Used in FedProto to compute class prototypes.
        """
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        """
        Returns probabilities (0-1) for evaluation/inference.
        """
        logits = self.forward(x)
        return torch.sigmoid(logits)


# Creating model instance
hospital2_model = Hospital2_MLP(
    input_size=9,
    dropout_rate=0.3
).to(device)

print("Hospital 2 Model (Coimbra - MLP) created!")
print(f"\nModel Architecture:")
print(hospital2_model)

# Parameter counts
total_params     = sum(
    p.numel() for p in hospital2_model.parameters()
)
trainable_params = sum(
    p.numel() for p in hospital2_model.parameters()
    if p.requires_grad
)
encoder_params   = sum(
    p.numel() for p in hospital2_model.encoder.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital2_model.shared_head.parameters()
)

print(f"\n{'='*45}")
print(f"  HOSPITAL 2 MODEL SUMMARY")
print(f"{'='*45}")
print(f"  Input features           : 9")
print(f"  Encoder Layer 1          : 9 → 32 units")
print(f"  Encoder Layer 2          : 32 → 64 units (embedding)")
print(f"  Shared head Layer 1      : 64 → 32 units")
print(f"  Shared head output       : 32 → 1 unit")
print(f"  Embedding dimension      : {EMBEDDING_DIM}")
print(f"  Dropout rate             : 0.3")
print(f"  Batch Normalization      : Yes")
print(f"  Total parameters         : {total_params:,}")
print(f"  Trainable parameters     : {trainable_params:,}")
print(f"  Encoder params           : {encoder_params:,} (private)")
print(f"  Shared head params       : {shared_head_params:,} (federated)")
print(f"  Activation (hidden)      : ReLU")
print(f"  Activation (output)      : None (BCEWithLogitsLoss)")
print(f"  FL component             : shared_head only")
print(f"{'='*45}")

Hospital 2 Model (Coimbra - MLP) created!

Model Architecture:
Hospital2_MLP(
  (encoder): Hospital2_Encoder(
    (fc1): Linear(in_features=9, out_features=32, bias=True)
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (fc2): Linear(in_features=32, out_features=64, bias=True)
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu2): ReLU()
    (dropout2): Dropout(p=0.3, inplace=False)
  )
  (shared_head): SharedClassificationHead(
    (fc1): Linear(in_features=64, out_features=32, bias=True)
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu1): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (output): Linear(in_features=32, out_features=1, bias=True)
  )
)

  HOSPITAL 2 MODEL SUMMARY
  Input features           : 9
  Encoder Layer 1          : 9 → 32 units
  Encoder Layer 

### 5. Test Hospital 2 Model

In [6]:
# Model Sanity Test — Hospital 2
# Tests both encoder and shared head separately
# as well as the full model end-to-end

hospital2_model.eval()

dummy_input = torch.randn(4, 9).to(device)

with torch.no_grad():

    # Test 1 — Full model forward pass
    logits        = hospital2_model(dummy_input)
    probabilities = hospital2_model.predict_proba(dummy_input)
    predictions   = (probabilities >= 0.5).float()

    # Test 2 — Encoder only (private component)
    embedding = hospital2_model.get_embedding(dummy_input)

    # Test 3 — Shared head only (federated component)
    shared_head_logits = hospital2_model.shared_head(embedding)

# Printing Full Test Results
print(f"\n{'='*55}")
print(f"  HOSPITAL 2 MODEL TEST — NEW ARCHITECTURE")
print(f"{'='*55}")

# Input
print(f"\n  Input:")
print(f"     Shape          : {dummy_input.shape}")
print(f"     Features       : 9 blood biomarker measurements")

# Encoder test
print(f"\n  Encoder (private — never shared):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {embedding.shape}")
print(f"     Embedding dim  : {embedding.shape[1]} (expected {EMBEDDING_DIM})")
print(f"     Embedding range: [{embedding.min().item():.4f}, "
      f"{embedding.max().item():.4f}]")
print(f"     All non-negative: "
      f"{'Yes' if embedding.min().item() >= 0 else 'No'} "
      f"(ReLU applied)")

embedding_dim_check = embedding.shape[1] == EMBEDDING_DIM
print(f"     Dim correct    : "
      f"{'Yes' if embedding_dim_check else 'No'} "
      f"(compatible with H1 and H3)")

# Shared head test
print(f"\n  Shared Head (federated — exchanged every round):")
print(f"     Input shape    : {embedding.shape}")
print(f"     Output shape   : {shared_head_logits.shape}")
print(f"     Raw logits     : "
      f"{shared_head_logits.squeeze().detach().cpu().numpy().round(4)}")

# Full model test
print(f"\n  Full Model (encoder → shared head):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {logits.shape}")
print(f"     Raw logits     : "
      f"{logits.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Probabilities  : "
      f"{probabilities.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Predictions    : "
      f"{predictions.squeeze().detach().cpu().numpy()}")
print(f"     Prob range     : [{probabilities.min().item():.4f}, "
      f"{probabilities.max().item():.4f}]")
print(f"     Expected range : [0.0000, 1.0000]")

range_check = (
    0.0 <= probabilities.min().item() and
    probabilities.max().item() <= 1.0
)
print(f"     Range valid    : "
      f"{'Yes' if range_check else 'No'}")

# Logit consistency check
logits_match = torch.allclose(logits, shared_head_logits, atol=1e-6)
print(f"\n  Consistency Check:")
print(f"     Full model logits == shared head logits: "
      f"{'Yes' if logits_match else 'No'}")
print(f"     (confirms encoder → shared head pipeline is intact)")

# Cross-hospital compatibility check
# Verifies H2 embedding is compatible with H1 shared head
with torch.no_grad():
    h2_embedding_through_h1_head = hospital1_model.shared_head(embedding)
    h1_accepts_h2_embedding = h2_embedding_through_h1_head.shape == torch.Size([4, 1])

print(f"\n  Cross-Hospital Compatibility Check:")
print(f"     H2 embedding → H1 shared head : "
      f"{'Compatible' if h1_accepts_h2_embedding else 'Incompatible'}")
print(f"     Output shape                  : "
      f"{h2_embedding_through_h1_head.shape}")
print(f"     (confirms shared 64-dim space works across hospitals)")

# Parameter check
encoder_params     = sum(
    p.numel() for p in hospital2_model.encoder.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital2_model.shared_head.parameters()
)
total_params       = sum(
    p.numel() for p in hospital2_model.parameters()
)

print(f"\n  Parameter Summary:")
print(f"     Encoder params     : {encoder_params:,} (private)")
print(f"     Shared head params : {shared_head_params:,} (federated)")
print(f"     Total params       : {total_params:,}")
print(f"     FL overhead        : {shared_head_params:,} params per round")

print(f"\n{'='*55}")

# Setting back to train mode
hospital2_model.train()
print(f"\nAll tests passed — Hospital 2 model working correctly!")


  HOSPITAL 2 MODEL TEST — NEW ARCHITECTURE

  Input:
     Shape          : torch.Size([4, 9])
     Features       : 9 blood biomarker measurements

  Encoder (private — never shared):
     Input shape    : torch.Size([4, 9])
     Output shape   : torch.Size([4, 64])
     Embedding dim  : 64 (expected 64)
     Embedding range: [0.0000, 0.9308]
     All non-negative: Yes (ReLU applied)
     Dim correct    : Yes (compatible with H1 and H3)

  Shared Head (federated — exchanged every round):
     Input shape    : torch.Size([4, 64])
     Output shape   : torch.Size([4, 1])
     Raw logits     : [-0.1398 -0.1063 -0.0856 -0.0772]

  Full Model (encoder → shared head):
     Input shape    : torch.Size([4, 9])
     Output shape   : torch.Size([4, 1])
     Raw logits     : [-0.1398 -0.1063 -0.0856 -0.0772]
     Probabilities  : [0.4651 0.4735 0.4786 0.4807]
     Predictions    : [0. 0. 0. 0.]
     Prob range     : [0.4651, 0.4807]
     Expected range : [0.0000, 1.0000]
     Range valid    : Ye

## PART C: HOSPITAL 3 MODEL (BreakHis - Images)

### 6. Define Hospital 3 CNN Model (MobileNetV2)

In [7]:
# HOSPITAL 3 — BREAKHIS IMAGE ENCODER
# Private component — never shared with central server
# Maps 224x224 RGB images → 64-dim embedding
# Uses pretrained MobileNetV2 as backbone
class Hospital3_Encoder(nn.Module):
    """
    Private local encoder for Hospital 3 BreakHis dataset.
    Input  : 224x224 RGB histopathology images
    Output : 64-dim shared embedding

    Architecture:
        MobileNetV2 backbone (frozen — ImageNet pretrained)
        → 1280-dim feature vector
        → Projection head: Linear(1280 → 64) + BN + ReLU + Dropout

    This component is NEVER shared in federated learning.
    It learns BreakHis-specific visual representations privately.

    Note: Only the projection head is trainable.
          MobileNetV2 backbone is completely frozen.
    """
    def __init__(self, dropout_rate=0.2):
        super(Hospital3_Encoder, self).__init__()

        # Loading pretrained MobileNetV2 backbone
        # weights=IMAGENET1K_V1 matches our normalization stats
        # mean=[0.485, 0.456, 0.406] std=[0.229, 0.224, 0.225]
        self.backbone = models.mobilenet_v2(
            weights=MobileNet_V2_Weights.IMAGENET1K_V1
        )

        # Freezing all backbone layers
        # ImageNet knowledge is preserved as fixed feature extractor
        # Only projection head below will be trained
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Removing MobileNetV2's original classifier entirely
        # Replacing it with an identity to get raw 1280-dim features
        self.backbone.classifier = nn.Identity()

        # Projection head: 1280 → 64-dim embedding
        # This is the only trainable part of the encoder
        # Maps MobileNetV2 features into shared embedding space
        self.projection = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, EMBEDDING_DIM),   # → 64-dim embedding
            nn.BatchNorm1d(EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        # Extracting 1280-dim features from frozen backbone
        features = self.backbone(x)          # shape: [batch, 1280]
        # Projecting to shared 64-dim embedding space
        embedding = self.projection(features) # shape: [batch, 64]
        return embedding


# HOSPITAL 3 — COMPLETE MODEL
# Combines private encoder with shared classification head
class Hospital3_CNN(nn.Module):
    """
    Complete model for Hospital 3 BreakHis dataset.
    Combines:
        - Private encoder   : MobileNetV2 + projection head
        - Shared head       : federated across all hospitals

    Input  : 224x224 RGB histopathology images
    Output : Binary logit (Benign vs Malignant)

    FL note: Only shared_head weights are exchanged.
             Encoder weights stay local and private always.
             MobileNetV2 backbone is frozen throughout.

    Note: Sigmoid is NOT applied here.
          BCEWithLogitsLoss handles it for numerical stability.
    """
    def __init__(self, dropout_rate=0.2):
        super(Hospital3_CNN, self).__init__()

        # Private — never shared
        self.encoder = Hospital3_Encoder(
            dropout_rate=dropout_rate
        )

        # Federated — shared every FL round
        # Same SharedClassificationHead as Hospital 1 and 2
        # Same architecture, same input dimension (64)
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)              # private encoding
        logit     = self.shared_head(embedding)  # shared classification
        return logit

    def get_embedding(self, x):
        """
        Returns 64-dim embedding before classification.
        Used in FedProto to compute class prototypes.
        """
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        """
        Returns probabilities (0-1) for evaluation/inference.
        """
        logits = self.forward(x)
        return torch.sigmoid(logits)


# Creating model instance
print("Loading pretrained MobileNetV2 weights...")
hospital3_model = Hospital3_CNN(dropout_rate=0.2).to(device)
print("Hospital 3 Model (BreakHis - MobileNetV2) created!")

# Parameter counts
total_params     = sum(
    p.numel() for p in hospital3_model.parameters()
)
trainable_params = sum(
    p.numel() for p in hospital3_model.parameters()
    if p.requires_grad
)
frozen_params      = total_params - trainable_params
encoder_params     = sum(
    p.numel() for p in hospital3_model.encoder.parameters()
)
projection_params  = sum(
    p.numel() for p in hospital3_model.encoder.projection.parameters()
)
backbone_params    = sum(
    p.numel() for p in hospital3_model.encoder.backbone.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital3_model.shared_head.parameters()
)

# Layer freezing verification
# Only projection head and shared head should be trainable
print(f"\nLayer freezing verification:")
trainable_layers = []
frozen_layers    = []
for name, param in hospital3_model.named_parameters():
    if param.requires_grad:
        trainable_layers.append(name)
    else:
        frozen_layers.append(name)

print(f"    Frozen layers    : {len(frozen_layers)} "
      f"(MobileNetV2 backbone)")
print(f"   Trainable layers : {len(trainable_layers)}")
print(f"\n   Trainable layer names:")
for name in trainable_layers:
    print(f"      - {name}")

# Model Summary
print(f"\n{'='*55}")
print(f"  HOSPITAL 3 MODEL SUMMARY")
print(f"{'='*55}")
print(f"  Base model               : MobileNetV2")
print(f"  Pretrained on            : ImageNet (IMAGENET1K_V1)")
print(f"  Input size               : 224x224 RGB")
print(f"  Backbone layers          : Frozen")
print(f"  Encoder projection       : Linear(1280→256→64)")
print(f"  Shared head              : Linear(64→32→1)")
print(f"  Embedding dimension      : {EMBEDDING_DIM}")
print(f"  Dropout rate             : 0.2")
print(f"  Batch Normalization      : Yes (projection + shared head)")
print(f"  Total parameters         : {total_params:,}")
print(f"  Frozen parameters        : {frozen_params:,} "
      f"(backbone)")
print(f"  Trainable parameters     : {trainable_params:,} "
      f"(projection + shared head)")
print(f"  Backbone params          : {backbone_params:,} "
      f"(private + frozen)")
print(f"  Projection params        : {projection_params:,} "
      f"(private + trainable)")
print(f"  Shared head params       : {shared_head_params:,} "
      f"(federated)")
print(f"  Activation (output)      : None (BCEWithLogitsLoss)")
print(f"  FL component             : shared_head only")
print(f"{'='*55}")

Loading pretrained MobileNetV2 weights...
Hospital 3 Model (BreakHis - MobileNetV2) created!

Layer freezing verification:
    Frozen layers    : 156 (MobileNetV2 backbone)
   Trainable layers : 14

   Trainable layer names:
      - encoder.projection.0.weight
      - encoder.projection.0.bias
      - encoder.projection.1.weight
      - encoder.projection.1.bias
      - encoder.projection.4.weight
      - encoder.projection.4.bias
      - encoder.projection.5.weight
      - encoder.projection.5.bias
      - shared_head.fc1.weight
      - shared_head.fc1.bias
      - shared_head.bn1.weight
      - shared_head.bn1.bias
      - shared_head.output.weight
      - shared_head.output.bias

  HOSPITAL 3 MODEL SUMMARY
  Base model               : MobileNetV2
  Pretrained on            : ImageNet (IMAGENET1K_V1)
  Input size               : 224x224 RGB
  Backbone layers          : Frozen
  Encoder projection       : Linear(1280→256→64)
  Shared head              : Linear(64→32→1)
  Embedding dim

### 7. Test Hospital 3 Model

In [8]:
# Model Sanity Test — Hospital 3
# Tests backbone, projection head and shared head separately
# as well as the full model end-to-end
hospital3_model.eval()

# Dummy input — batch of 4 RGB images of size 224x224
dummy_input = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():

    # Test 1 — Full model forward pass
    logits        = hospital3_model(dummy_input)
    probabilities = hospital3_model.predict_proba(dummy_input)
    predictions   = (probabilities >= 0.5).float()

    # Test 2 — Backbone only (frozen MobileNetV2)
    backbone_features = hospital3_model.encoder.backbone(dummy_input)

    # Test 3 — Encoder only (backbone + projection)
    embedding = hospital3_model.get_embedding(dummy_input)

    # Test 4 — Projection head only
    projection_output = hospital3_model.encoder.projection(
        backbone_features
    )

    # Test 5 — Shared head only (federated component)
    shared_head_logits = hospital3_model.shared_head(embedding)

# Printing Full Test Results
print(f"\n{'='*55}")
print(f"  HOSPITAL 3 MODEL TEST — NEW ARCHITECTURE")
print(f"{'='*55}")

# Input
print(f"\n  Input:")
print(f"     Shape          : {dummy_input.shape}")
print(f"     Type           : RGB histopathology images")
print(f"     Normalized     : ImageNet mean/std")

# Backbone test (frozen MobileNetV2)
print(f"\n   Backbone (frozen MobileNetV2 — private):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {backbone_features.shape}")
print(f"     Feature dim    : {backbone_features.shape[1]} "
      f"(expected 1280)")
print(f"     Frozen         :  Yes — no gradients computed")

backbone_dim_check = backbone_features.shape[1] == 1280
print(f"     Dim correct    : "
      f"{'Yes' if backbone_dim_check else 'No'}")

# Projection head test
print(f"\n  Projection Head (private — trainable):")
print(f"     Input shape    : {backbone_features.shape}")
print(f"     Output shape   : {projection_output.shape}")
print(f"     Projects       : 1280 → 256 → 64")
print(f"     All non-negative: "
      f"{'Yes' if projection_output.min().item() >= 0 else 'No'} "
      f"(ReLU applied)")

# Encoder test (backbone + projection combined)
print(f"\n  Encoder (backbone + projection — never shared):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {embedding.shape}")
print(f"     Embedding dim  : {embedding.shape[1]} "
      f"(expected {EMBEDDING_DIM})")
print(f"     Embedding range: [{embedding.min().item():.4f}, "
      f"{embedding.max().item():.4f}]")
print(f"     All non-negative: "
      f"{'Yes' if embedding.min().item() >= 0 else 'No'} "
      f"(ReLU applied)")

embedding_dim_check = embedding.shape[1] == EMBEDDING_DIM
print(f"     Dim correct    : "
      f"{'Yes' if embedding_dim_check else 'No'} "
      f"(compatible with H1 and H2)")

# Shared head test
print(f"\n  Shared Head (federated — exchanged every round):")
print(f"     Input shape    : {embedding.shape}")
print(f"     Output shape   : {shared_head_logits.shape}")
print(f"     Raw logits     : "
      f"{shared_head_logits.squeeze().detach().cpu().numpy().round(4)}")

# Full model test
print(f"\n  Full Model (encoder → shared head):")
print(f"     Input shape    : {dummy_input.shape}")
print(f"     Output shape   : {logits.shape}")
print(f"     Raw logits     : "
      f"{logits.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Probabilities  : "
      f"{probabilities.squeeze().detach().cpu().numpy().round(4)}")
print(f"     Predictions    : "
      f"{predictions.squeeze().detach().cpu().numpy()}")
print(f"     Prob range     : [{probabilities.min().item():.4f}, "
      f"{probabilities.max().item():.4f}]")
print(f"     Expected range : [0.0000, 1.0000]")

range_check = (
    0.0 <= probabilities.min().item() and
    probabilities.max().item() <= 1.0
)
print(f"     Range valid    : "
      f"{'Yes' if range_check else 'No'}")

# Logit consistency check
logits_match = torch.allclose(logits, shared_head_logits, atol=1e-6)
print(f"\n  Consistency Check:")
print(f"     Full model logits == shared head logits: "
      f"{'Yes' if logits_match else 'No'}")
print(f"     (confirms encoder → shared head pipeline is intact)")

# Cross-hospital compatibility check
# Feeds H3 embedding through H1 and H2 shared heads
# Proves all three hospitals share the same 64-dim space
with torch.no_grad():
    h3_through_h1 = hospital1_model.shared_head(embedding)
    h3_through_h2 = hospital2_model.shared_head(embedding)

    h1_compatible = h3_through_h1.shape == torch.Size([4, 1])
    h2_compatible = h3_through_h2.shape == torch.Size([4, 1])

print(f"\n  Cross-Hospital Compatibility Check:")
print(f"     H3 embedding → H1 shared head : "
      f"{'Compatible' if h1_compatible else 'Incompatible'} "
      f"— shape {h3_through_h1.shape}")
print(f"     H3 embedding → H2 shared head : "
      f"{'Compatible' if h2_compatible else 'Incompatible'} "
      f"— shape {h3_through_h2.shape}")
print(f"     (confirms all three hospitals share 64-dim space)")
print(f"     (FedAvg is now mathematically valid )")

# Gradient flow check
# Confirms backbone is truly frozen
# and projection + shared head are trainable
print(f"\n  Gradient Flow Check:")

backbone_trainable = any(
    p.requires_grad
    for p in hospital3_model.encoder.backbone.parameters()
)
projection_trainable = any(
    p.requires_grad
    for p in hospital3_model.encoder.projection.parameters()
)
shared_head_trainable = any(
    p.requires_grad
    for p in hospital3_model.shared_head.parameters()
)

print(f"     Backbone trainable    : "
      f"{'Yes — should be frozen!' if backbone_trainable else 'No — correctly frozen'}")
print(f"     Projection trainable  : "
      f"{'Yes — correct' if projection_trainable else 'No — should be trainable!'}")
print(f"     Shared head trainable : "
      f"{'Yes — correct' if shared_head_trainable else 'No — should be trainable!'}")

# Parameter summary
total_params       = sum(
    p.numel() for p in hospital3_model.parameters()
)
trainable_params   = sum(
    p.numel() for p in hospital3_model.parameters()
    if p.requires_grad
)
frozen_params      = total_params - trainable_params
backbone_params    = sum(
    p.numel() for p in
    hospital3_model.encoder.backbone.parameters()
)
projection_params  = sum(
    p.numel() for p in
    hospital3_model.encoder.projection.parameters()
)
shared_head_params = sum(
    p.numel() for p in hospital3_model.shared_head.parameters()
)

print(f"\n  Parameter Summary:")
print(f"     Backbone params    : {backbone_params:,} "
      f"(private + frozen)")
print(f"     Projection params  : {projection_params:,} "
      f"(private + trainable)")
print(f"     Shared head params : {shared_head_params:,} "
      f"(federated)")
print(f"     Total params       : {total_params:,}")
print(f"     Trainable params   : {trainable_params:,}")
print(f"     Frozen params      : {frozen_params:,}")
print(f"     FL overhead        : {shared_head_params:,} "
      f"params per round")

print(f"\n{'='*55}")

# Set back to train mode
hospital3_model.train()
print(f"\nAll tests passed — Hospital 3 model working correctly!")


  HOSPITAL 3 MODEL TEST — NEW ARCHITECTURE

  Input:
     Shape          : torch.Size([4, 3, 224, 224])
     Type           : RGB histopathology images
     Normalized     : ImageNet mean/std

   Backbone (frozen MobileNetV2 — private):
     Input shape    : torch.Size([4, 3, 224, 224])
     Output shape   : torch.Size([4, 1280])
     Feature dim    : 1280 (expected 1280)
     Frozen         :  Yes — no gradients computed
     Dim correct    : Yes

  Projection Head (private — trainable):
     Input shape    : torch.Size([4, 1280])
     Output shape   : torch.Size([4, 64])
     Projects       : 1280 → 256 → 64
     All non-negative: Yes (ReLU applied)

  Encoder (backbone + projection — never shared):
     Input shape    : torch.Size([4, 3, 224, 224])
     Output shape   : torch.Size([4, 64])
     Embedding dim  : 64 (expected 64)
     Embedding range: [0.0000, 0.3556]
     All non-negative: Yes (ReLU applied)
     Dim correct    : Yes (compatible with H1 and H2)

  Shared Head (feder

## SAVE MODELS

### 8. Save Model Architectures

In [9]:
# STEP 1 — Creating models directory
os.makedirs('models', exist_ok=True)
print("Models directory ready")

# STEP 2 — Saving updated model architecture definitions
model_code = '''# -*- coding: utf-8 -*-
"""
Model Architectures for Federated Learning Breast Cancer Detection
Three hospital models for privacy-preserving federated learning.

Hospitals:
    Hospital 1 - WDBC      : MLP (23 tabular features)
    Hospital 2 - Coimbra   : MLP (9 blood biomarker features)
    Hospital 3 - BreakHis  : MobileNetV2 CNN (224x224 images)

Architecture:
    Each hospital model consists of two components:
        1. Private Encoder   - modality-specific, never shared
        2. Shared Head       - identical across all hospitals,
                               federated via FedAvg every round

    This design ensures FedAvg is mathematically valid across
    heterogeneous data types (tabular + image).

Note:
    Sigmoid is NOT included in forward() for any model.
    BCEWithLogitsLoss handles sigmoid during training.
    Use predict_proba() for inference probabilities.
    Use get_embedding() for FedProto prototype computation.
"""

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import MobileNet_V2_Weights


# ============================================================
# SHARED CONSTANT
# All hospitals must use the same embedding dimension
# for FedAvg to be mathematically valid
# ============================================================
EMBEDDING_DIM = 64


# ============================================================
# SHARED CLASSIFICATION HEAD
# Identical architecture across ALL three hospitals
# This is the ONLY component exchanged in federated learning
# Input  : 64-dim embedding (from any modality encoder)
# Output : Binary logit (Benign vs Malignant)
# ============================================================
class SharedClassificationHead(nn.Module):
    """
    Shared federated classification head.
    Used by all three hospitals.
    Input  : 64-dim embedding
    Output : Binary logit - no sigmoid
    """
    def __init__(self, embedding_dim=EMBEDDING_DIM, dropout_rate=0.3):
        super(SharedClassificationHead, self).__init__()
        self.fc1      = nn.Linear(embedding_dim, 32)
        self.bn1      = nn.BatchNorm1d(32)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.output   = nn.Linear(32, 1)

    def forward(self, embedding):
        x = self.fc1(embedding)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.output(x)
        return x


# ============================================================
# HOSPITAL 1 - WDBC ENCODER
# Private - never shared
# Input  : 23 cell nucleus features
# Output : 64-dim shared embedding
# ============================================================
class Hospital1_Encoder(nn.Module):
    """Private encoder for Hospital 1 WDBC dataset."""
    def __init__(self, input_size=23, dropout_rate=0.3):
        super(Hospital1_Encoder, self).__init__()
        self.fc1      = nn.Linear(input_size, 128)
        self.bn1      = nn.BatchNorm1d(128)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2      = nn.Linear(128, EMBEDDING_DIM)
        self.bn2      = nn.BatchNorm1d(EMBEDDING_DIM)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        return x


# ============================================================
# HOSPITAL 1 - COMPLETE MODEL
# Encoder (private) + SharedHead (federated)
# ============================================================
class Hospital1_MLP(nn.Module):
    """Complete model for Hospital 1 WDBC dataset."""
    def __init__(self, input_size=23, dropout_rate=0.3):
        super(Hospital1_MLP, self).__init__()
        self.encoder     = Hospital1_Encoder(
            input_size=input_size,
            dropout_rate=dropout_rate
        )
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)
        logit     = self.shared_head(embedding)
        return logit

    def get_embedding(self, x):
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))


# ============================================================
# HOSPITAL 2 - COIMBRA ENCODER
# Private - never shared
# Input  : 9 blood biomarker features
# Output : 64-dim shared embedding
# ============================================================
class Hospital2_Encoder(nn.Module):
    """Private encoder for Hospital 2 Coimbra dataset."""
    def __init__(self, input_size=9, dropout_rate=0.3):
        super(Hospital2_Encoder, self).__init__()
        self.fc1      = nn.Linear(input_size, 32)
        self.bn1      = nn.BatchNorm1d(32)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2      = nn.Linear(32, EMBEDDING_DIM)
        self.bn2      = nn.BatchNorm1d(EMBEDDING_DIM)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        return x


# ============================================================
# HOSPITAL 2 - COMPLETE MODEL
# Encoder (private) + SharedHead (federated)
# ============================================================
class Hospital2_MLP(nn.Module):
    """Complete model for Hospital 2 Coimbra dataset."""
    def __init__(self, input_size=9, dropout_rate=0.3):
        super(Hospital2_MLP, self).__init__()
        self.encoder     = Hospital2_Encoder(
            input_size=input_size,
            dropout_rate=dropout_rate
        )
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)
        logit     = self.shared_head(embedding)
        return logit

    def get_embedding(self, x):
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))


# ============================================================
# HOSPITAL 3 - BREAKHIS IMAGE ENCODER
# Private - never shared
# Input  : 224x224 RGB histopathology images
# Output : 64-dim shared embedding
# ============================================================
class Hospital3_Encoder(nn.Module):
    """Private encoder for Hospital 3 BreakHis dataset."""
    def __init__(self, dropout_rate=0.2):
        super(Hospital3_Encoder, self).__init__()

        # Frozen MobileNetV2 backbone
        self.backbone = models.mobilenet_v2(
            weights=MobileNet_V2_Weights.IMAGENET1K_V1
        )
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.classifier = nn.Identity()

        # Trainable projection head: 1280 -> 256 -> 64
        self.projection = nn.Sequential(
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, EMBEDDING_DIM),
            nn.BatchNorm1d(EMBEDDING_DIM),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.projection(features)
        return embedding


# ============================================================
# HOSPITAL 3 - COMPLETE MODEL
# Encoder (private) + SharedHead (federated)
# ============================================================
class Hospital3_CNN(nn.Module):
    """Complete model for Hospital 3 BreakHis dataset."""
    def __init__(self, dropout_rate=0.2):
        super(Hospital3_CNN, self).__init__()
        self.encoder     = Hospital3_Encoder(
            dropout_rate=dropout_rate
        )
        self.shared_head = SharedClassificationHead(
            embedding_dim=EMBEDDING_DIM,
            dropout_rate=dropout_rate
        )

    def forward(self, x):
        embedding = self.encoder(x)
        logit     = self.shared_head(embedding)
        return logit

    def get_embedding(self, x):
        with torch.no_grad():
            return self.encoder(x)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))
'''

# Saving architecture file with utf-8 encoding
with open('models/model_architectures.py', 'w', encoding='utf-8') as f:
    f.write(model_code)
print("Model architectures saved: models/model_architectures.py")

# STEP 3 — Verifying model structure before saving weights
# ============================================================
print("\nVerifying model structure before saving...")

assert hasattr(hospital1_model, 'encoder'), \
    "H1 missing encoder"
assert hasattr(hospital1_model, 'shared_head'), \
    "H1 missing shared_head"
assert hasattr(hospital2_model, 'encoder'), \
    "H2 missing encoder"
assert hasattr(hospital2_model, 'shared_head'), \
    "H2 missing shared_head"
assert hasattr(hospital3_model, 'encoder'), \
    "H3 missing encoder"
assert hasattr(hospital3_model, 'shared_head'), \
    "H3 missing shared_head"

print("   Hospital 1 — encoder + shared_head confirmed")
print("   Hospital 2 — encoder + shared_head confirmed")
print("   Hospital 3 — encoder + shared_head confirmed")

# STEP 4 — Saving initial untrained weights
#          Saving full model + components separately
#          Components saved separately for FL use

# Hospital 1
torch.save(
    hospital1_model.state_dict(),
    'models/hospital1_initial.pth'
)
torch.save(
    hospital1_model.encoder.state_dict(),
    'models/hospital1_encoder_initial.pth'
)
torch.save(
    hospital1_model.shared_head.state_dict(),
    'models/hospital1_shared_head_initial.pth'
)

# Hospital 2
torch.save(
    hospital2_model.state_dict(),
    'models/hospital2_initial.pth'
)
torch.save(
    hospital2_model.encoder.state_dict(),
    'models/hospital2_encoder_initial.pth'
)
torch.save(
    hospital2_model.shared_head.state_dict(),
    'models/hospital2_shared_head_initial.pth'
)

# Hospital 3
torch.save(
    hospital3_model.state_dict(),
    'models/hospital3_initial.pth'
)
torch.save(
    hospital3_model.encoder.state_dict(),
    'models/hospital3_encoder_initial.pth'
)
torch.save(
    hospital3_model.shared_head.state_dict(),
    'models/hospital3_shared_head_initial.pth'
)

print("Initial model weights saved")

# STEP 5 — Verifying all saved files are loadable
print("\nVerifying saved files...")

# Verifying .py file
try:
    with open(
        'models/model_architectures.py', 'r', encoding='utf-8'
    ) as f:
        content = f.read()
    assert 'EMBEDDING_DIM'              in content
    assert 'SharedClassificationHead'   in content
    assert 'Hospital1_Encoder'          in content
    assert 'Hospital1_MLP'              in content
    assert 'Hospital2_Encoder'          in content
    assert 'Hospital2_MLP'              in content
    assert 'Hospital3_Encoder'          in content
    assert 'Hospital3_CNN'              in content
    print("   model_architectures.py  — valid and readable")
    print(f"      Contains all 8 required classes/constants")
except Exception as e:
    print(f"   model_architectures.py  — error: {e}")

# Verifying all .pth files
pth_files = [
    ('hospital1_initial.pth',            'Hospital 1 full model'),
    ('hospital1_encoder_initial.pth',    'Hospital 1 encoder'),
    ('hospital1_shared_head_initial.pth','Hospital 1 shared head'),
    ('hospital2_initial.pth',            'Hospital 2 full model'),
    ('hospital2_encoder_initial.pth',    'Hospital 2 encoder'),
    ('hospital2_shared_head_initial.pth','Hospital 2 shared head'),
    ('hospital3_initial.pth',            'Hospital 3 full model'),
    ('hospital3_encoder_initial.pth',    'Hospital 3 encoder'),
    ('hospital3_shared_head_initial.pth','Hospital 3 shared head'),
]

for filename, description in pth_files:
    try:
        state = torch.load(
            f'models/{filename}',
            map_location='cpu'
        )
        print(f"   {filename:<40} "
              f"— {len(state)} layers")
    except Exception as e:
        print(f"   {filename:<40} "
              f"— error: {e}")

# STEP 6 — Generating Final Summary
h1_total = sum(p.numel() for p in hospital1_model.parameters())
h2_total = sum(p.numel() for p in hospital2_model.parameters())
h3_total = sum(p.numel() for p in hospital3_model.parameters())
h1_head  = sum(
    p.numel() for p in hospital1_model.shared_head.parameters()
)
h2_head  = sum(
    p.numel() for p in hospital2_model.shared_head.parameters()
)
h3_head  = sum(
    p.numel() for p in hospital3_model.shared_head.parameters()
)

print(f"\n{'='*55}")
print(f"  MODEL SAVING SUMMARY")
print(f"{'='*55}")
print(f"  Architecture file    : models/model_architectures.py")
print(f"  Embedding dimension  : {EMBEDDING_DIM}")
print(f"{'─'*55}")
print(f"  {'Model':<12} {'Total':>10} {'FL Params':>12} "
      f"{'Saved Files':>5}")
print(f"{'─'*55}")
print(f"  {'Hospital 1':<12} {h1_total:>10,} {h1_head:>12,}   3 files")
print(f"  {'Hospital 2':<12} {h2_total:>10,} {h2_head:>12,}   3 files")
print(f"  {'Hospital 3':<12} {h3_total:>10,} {h3_head:>12,}   3 files")
print(f"{'─'*55}")
print(f"  FL params = shared_head params exchanged per round")
print(f"  All three shared heads : identical {h1_head:,} params")
print(f"  Weights status         : Untrained initial weights")
print(f"  Purpose                : Baseline before FL training")
print(f"{'='*55}")

Models directory ready
Model architectures saved: models/model_architectures.py

Verifying model structure before saving...
   Hospital 1 — encoder + shared_head confirmed
   Hospital 2 — encoder + shared_head confirmed
   Hospital 3 — encoder + shared_head confirmed
Initial model weights saved

Verifying saved files...
   model_architectures.py  — valid and readable
      Contains all 8 required classes/constants
   hospital1_initial.pth                    — 23 layers
   hospital1_encoder_initial.pth            — 14 layers
   hospital1_shared_head_initial.pth        — 9 layers
   hospital2_initial.pth                    — 23 layers
   hospital2_encoder_initial.pth            — 14 layers
   hospital2_shared_head_initial.pth        — 9 layers
   hospital3_initial.pth                    — 335 layers
   hospital3_encoder_initial.pth            — 326 layers
   hospital3_shared_head_initial.pth        — 9 layers

  MODEL SAVING SUMMARY
  Architecture file    : models/model_architectures.py


## FINAL SUMMARY

### 9. Phase 2 Summary

In [ ]:
print("\n" + "="*70)
print("  PHASE 2 COMPLETE — ALL MODELS BUILT!")
print("="*70)

# Computing parameter counts

# Hospital 1
h1_total      = sum(p.numel() for p in hospital1_model.parameters())
h1_trainable  = sum(
    p.numel() for p in hospital1_model.parameters()
    if p.requires_grad
)
h1_output     = sum(
    p.numel() for p in hospital1_model.output_layer.parameters()
)

# Hospital 2
h2_total      = sum(p.numel() for p in hospital2_model.parameters())
h2_trainable  = sum(
    p.numel() for p in hospital2_model.parameters()
    if p.requires_grad
)
h2_output     = sum(
    p.numel() for p in hospital2_model.output_layer.parameters()
)

# Hospital 3
h3_total      = sum(p.numel() for p in hospital3_model.parameters())
h3_trainable  = sum(
    p.numel() for p in hospital3_model.parameters()
    if p.requires_grad
)
h3_frozen     = h3_total - h3_trainable
h3_output     = sum(
    p.numel() for p in hospital3_model.output_layer.parameters()
)

summary = f"""
** Hospital 1 Model (WDBC — Cell Nucleus Features):
   Architecture    : MLP (23 → 64 → 32 → 1)
   Input           : 23 cell nucleus features (7 dropped, correlation > 0.95)
   Hidden layers   : 64 units + BatchNorm + ReLU + Dropout(0.3)
                     32 units + BatchNorm + ReLU + Dropout(0.3)
   Total params    : {h1_total:,}
   Trainable params: {h1_trainable:,}
   Output layer    : {h1_output} params (shared in FL)
   Activation      : ReLU (hidden) — BCEWithLogitsLoss (output)
   Class balance   : SMOTE applied

** Hospital 2 Model (Coimbra — Blood Biomarkers):
   Architecture    : MLP (9 → 32 → 16 → 1)
   Input           : 9 blood biomarker features
   Hidden layers   : 32 units + BatchNorm + ReLU + Dropout(0.3)
                     16 units + BatchNorm + ReLU + Dropout(0.3)
   Total params    : {h2_total:,}
   Trainable params: {h2_trainable:,}
   Output layer    : {h2_output} params (shared in FL)
   Activation      : ReLU (hidden) — BCEWithLogitsLoss (output)
   Class balance   : SMOTE applied

** Hospital 3 Model (BreakHis — Histopathology Images):
   Architecture    : MobileNetV2 (pretrained) + Custom Output Layer
   Input           : 224x224 RGB histopathology images
   Output layer    : Dropout(0.2) + Linear(1280 → 1)
   Total params    : {h3_total:,}
   Frozen params   : {h3_frozen:,} (MobileNetV2 base — locked)
   Trainable params: {h3_trainable:,} (output layer only)
   Output layer    : {h3_output} params (shared in FL)
   Activation      : BCEWithLogitsLoss (output)
   Transfer learn  : ImageNet weights — IMAGENET1K_V1
   Class balance   : WeightedRandomSampler

** Key Points:
   - All models output raw logits — NOT probabilities
   - Sigmoid applied by BCEWithLogitsLoss during training
   - Use predict_proba() for probability output during inference
   - Each model has an output_layer shared in federated learning
   - Hospital 3 uses transfer learning — only 0.057% params trained
   - Data leakage prevented — SMOTE and scaling on train only
   - Device: {device}

** Saved Files:
   - models/model_architectures.py  (updated model definitions)
   - models/hospital1_initial.pth   (untrained weights)
   - models/hospital2_initial.pth   (untrained weights)
   - models/hospital3_initial.pth   (untrained weights)
   - models/phase2_summary.txt      (this summary)

** Next Step: Phase 3 — Train Local Models Independently
"""

print(summary)
print("="*70)

# Save summary with verification
try:
    with open('models/phase2_summary.txt', 'w', encoding='utf-8') as f:
        f.write(summary)
    # Verify it was saved correctly
    with open('models/phase2_summary.txt', 'r') as f:
        saved = f.read()
    assert 'Hospital 1' in saved
    assert 'Hospital 2' in saved
    assert 'Hospital 3' in saved
    print("Summary saved and verified: models/phase2_summary.txt")
except Exception as e:
    print(f"Summary save failed: {e}")